# Setup
Notebooks call reusable functions from `src/`.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import load_config, resolve_paths, set_global_seed, get_seed
config = load_config(ROOT / 'configs/project_config.yaml')
paths = resolve_paths(config)
set_global_seed(get_seed(config))
print('project root:', paths.root)


## XGBoost training

In [ ]:
from src.data_loader import load_parquet
from src.feature_engineering import get_model_feature_columns
from src.temporal_split import chronological_date_split, masks_from_split
from src.train_xgboost import train_xgboost
df = load_parquet(paths.processed).sort_values('timestamp').reset_index(drop=True)
split = chronological_date_split(df, 0.6, 0.2, 0.2, 60)
masks = masks_from_split(len(df), split)
valid = df['label'].notna().to_numpy()
for k in masks: masks[k] &= valid
cols = get_model_feature_columns(df)
res = train_xgboost(df, cols, masks, config, paths.models)
print('best', res['best_params'])
print('val', res['metrics_val']['macro_f1'], 'test', res['metrics_test']['macro_f1'])
